<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c03-header.png" alt="Nextia Learning · Essential Mathematics and Statistics for AI" width="100%">

# Solution: Events and uncertainty

**[C03-M03-L01 · Events and uncertainty](https://learning.nextia-ai.com/courses/math/m03/events-and-uncertainty/)** · C03, Essential Mathematics and Statistics for AI · Module 3, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**Optional: check the numbers in Python.** This notebook is optional. The lesson works without code: a calculator is enough. Use the notebook to check the lesson's numbers and to redo its worked examples and tasks in Python.

**You will** count 100 Tuesday tickets to find probabilities, complements, "and" and "or", test two events for independence, and find the chance that at least one of three checks fails, with the formula and with a seeded simulation.

| | |
|---|---|
| **Time** | About 25 minutes, after you read the lesson |
| **Needs** | Nothing to install and no account: only Python's standard library. It runs in Colab, Kaggle or any Jupyter. |
| **You should know** | How to change a fraction into a decimal and a percentage (8/100 = 0.08 = 8%), from [Mathematical reading skills: Fractions, decimals and percentages](https://learning.nextia-ai.com/courses/math/m01/mathematical-reading-skills/#fractions-decimals-and-percentages). |
| **Tested** | 2026-10-07, Python 3.14.6 (standard library only), in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

The [lesson page](https://learning.nextia-ai.com/courses/math/m03/events-and-uncertainty/) has the full explanations, the charts, the explorers and the knowledge checks that count toward your certificate. Read a section there, then check its numbers here.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, replace each `...` with your calculation, run the cell, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C03/M03-L01-events-and-uncertainty/events-and-uncertainty-solution.ipynb).

## Setup: the helpers

The next cell defines three small functions that the notebook uses. You do not need to read them, but you can.

- `expect(what, yours, expected)` compares your number with the lesson's number. It says "Check passed." or if yours is too high or too low. It does not show the answer.
- `table(headers, rows)` prints a small table with aligned columns.
- `bars(labels, counts)` prints a text bar chart: one row of `█` for each label.

Run the cell. You should see `Ready.`

In [ ]:
# Helpers: check an answer without showing it, print a table, draw text bars.
import math, random, statistics

def expect(what, yours, expected, tol=0.01):
    """Compare your value with the expected one, to within `tol` for numbers."""
    if yours is ...:
        print(f"Not yet: replace ... with your calculation for {what}, then run the cell again.")
        return
    if isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        if abs(yours - expected) <= tol:
            print(f"Check passed: {what} is {yours:g}.")
        else:
            side = "too high" if yours > expected else "too low"
            print(f"Not yet: your {what}, {yours:g}, is {side}. Look at the step before it again.")
    elif yours == expected:
        print(f"Check passed: {what} is {yours}.")
    else:
        print(f"Not yet: your {what} is {yours!r}. That is not the expected answer.")

def table(headers, rows, digits=2):
    """Print rows as a table. Floats are rounded to `digits` decimals."""
    cell = lambda v: f"{v:,.{digits}f}" if isinstance(v, float) else str(v)
    text = [[cell(v) for v in row] for row in rows]
    widths = [max(len(str(h)), *(len(r[i]) for r in text)) for i, h in enumerate(headers)]
    print("  ".join(str(h).rjust(w) for h, w in zip(headers, widths)))
    for r in text:
        print("  ".join(v.rjust(w) for v, w in zip(r, widths)))

def bars(labels, counts, width=40):
    """Print a text bar chart: the longest bar is `width` characters."""
    top = max(counts) or 1
    size = max(len(str(label)) for label in labels)
    for label, n in zip(labels, counts):
        print(f"{str(label).rjust(size)} | {'█' * round(n / top * width)} {n}")

print("Ready.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Count first: 100 tickets from one Tuesday

Priya takes 100 tickets from one Tuesday. For each team, she counts the tickets and the urgent tickets. A **probability** is the share of cases in which something happens: count the cases, then divide by all cases.

Run the cell. You should see the table, the total `100` tickets and `8` urgent tickets, and `P(urgent) = 0.08`.

In [ ]:
teams = ["Returns", "Delivery", "Payment", "Warranty", "Account"]
tickets = [34, 28, 15, 13, 10]   # tickets for each team
urgent = [1, 2, 4, 0, 1]         # urgent tickets for each team

table(["Team", "Tickets", "Urgent"], [[t, n, u] for t, n, u in zip(teams, tickets, urgent)])
total = sum(tickets)
print()
print("total tickets:", total)
print("urgent tickets:", sum(urgent))
print("P(urgent) =", sum(urgent) / total)

The **sample space** is the list of all possible outcomes: here, the five teams. Exactly one team gets each ticket, so the five probabilities add up to 1.

> **Predict.** What is the sum of the five probabilities? Write it down, then run the cell.

In [ ]:
p_team = [n / total for n in tickets]
table(["Team", "Probability"], [[t, p] for t, p in zip(teams, p_team)])
print()
print("sum of the probabilities:", sum(tickets) / total)
bars(teams, tickets)

> **Check.** You should see `sum of the probabilities: 1.0`, and the bars from Returns (34) down to Account (10). The cell divides the sum of the counts (100) by 100. If you add the five decimals instead, Python can print a number such as `0.9999999999999999`, because it stores most decimals with a very small error.

## 2. Complements: "not" is often easier

The **complement** of an event is "the event does not happen": P(not urgent) = 1 − P(urgent).

Run the cell. You should see `P(not urgent) = 0.92`, the same value two ways.

In [ ]:
p_urgent = sum(urgent) / total
print("P(not urgent) =", 1 - p_urgent)
print("count it:      ", (total - sum(urgent)) / total)

## 3. Joint events: "and" and "or"

The lesson's grid shows the 100 tickets: 15 payment tickets, 8 urgent tickets and 4 urgent payment tickets. For "or", the 4 urgent payment tickets are in both groups, so subtract them once.

> **Predict.** What is P(payment or urgent)? Then run the cell.

In [ ]:
payment, urg, both = 15, 8, 4

rows = [
    ["Payment ticket", payment, payment / total],
    ["Urgent", urg, urg / total],
    ["Payment and urgent", both, both / total],
    ["Payment or urgent", payment + urg - both, (payment + urg - both) / total],
]
table(["Event", "Count", "Probability"], rows)

> **Check.** You should see 15 + 8 − 4 = `19` tickets and a probability of `0.19`. If you add 15 and 8 without the subtraction, you count the 4 urgent payment tickets twice.

## 4. Independence: does one event change the other?

If "payment ticket" and "urgent" were **independent**, payment tickets would be urgent as often as all tickets: 8 in 100. Then you expect 15 × 0.08 urgent payment tickets.

Run the cell. You should see `expected if independent: 1.2`, `real count: 4`, and the multiplication rule giving `0.012` instead of the real `0.04`.

In [ ]:
expected = payment * p_urgent
print("expected if independent:", round(expected, 2))
print("real count:             ", both)
print()
print("P(payment) x P(urgent):    ", round(payment / total * p_urgent, 3))
print("real P(payment and urgent):", both / total)

> **Check.** Payment tickets are urgent much more often than other tickets. So the two events are **not** independent, and the multiplication rule gives the wrong answer here.

## 5. A model's output is a probability over a sample space

The team model reads ticket T-2207, "My card was charged twice for one order", and gives one number for each team.

Run the cell. You should see a sum of `1.0`, P(not payment) `0.19` and P(account or returns) `0.15`.

In [ ]:
scores = {"Payment": 0.81, "Returns": 0.09, "Account": 0.06, "Delivery": 0.03, "Warranty": 0.01}

print("sum of the outputs:   ", round(sum(scores.values()), 2))
print("P(not payment):        ", round(1 - scores["Payment"], 2))
print("P(account or returns):", round(scores["Account"] + scores["Returns"], 2))

The cell rounds to 2 decimals, because Python stores 0.81, 0.09 and the others with a very small error. A ticket goes to exactly one team, so "account" and "returns" cannot both happen: nothing is counted twice.

## 6. At least one of three checks fails

Each ticket passes three checks. Each check fails for 2 in 100 tickets. "At least one fails" has many cases; its complement, "none fails", has one. If the checks fail independently, multiply the chances that each one works.

> **Predict.** About how many of 1,200 tickets in a week have at least one failure? Then run the cell.

In [ ]:
fail = 0.02
one_works = 1 - fail
all_work = one_works ** 3
at_least_one = 1 - all_work

table(["Step", "Result"], [
    ["One check works", one_works],
    ["All three work", all_work],
    ["At least one fails", at_least_one],
    ["Tickets in a week of 1,200", 1200 * at_least_one],
], digits=4)

> **Check.** You should see `0.9800`, `0.9412`, `0.0588` and `70.5696` tickets: about 71 tickets a week, or 5.9%. The lesson rounds "all three work" to 0.941.

The lesson's line chart shows the chance of at least one failure for 1 to 10 checks. The next cell prints the same values as a table, in percent. Run it. You should see 5.9 for 3 checks at 2%, and 14.3 for 3 checks at 5%.

In [ ]:
rows = []
for k in range(1, 11):
    rows.append([k, round((1 - 0.98 ** k) * 100, 1), round((1 - 0.95 ** k) * 100, 1)])
table(["Checks", "Each fails 2% (%)", "Each fails 5% (%)"], rows, digits=1)

## 7. Check the formula with a simulation

A **simulation** makes random tickets with a program and counts what happens. This is the lesson's optional Python box. `random.seed(1)` fixes the random numbers, so you get the same result every time you run it.

> **Predict.** Is the simulated share exactly 0.0588? Then run the cell. It takes a second or two.

In [ ]:
random.seed(1)
FAIL = 0.02        # each check fails for 2 in 100 tickets
TICKETS = 100_000

with_failure = 0
for ticket in range(TICKETS):
    fails = [random.random() < FAIL for check in range(3)]
    if any(fails):
        with_failure += 1

print("Share with at least one failure:", with_failure / TICKETS)
print("Formula 1 - 0.98 ** 3:", round(1 - 0.98 ** 3, 4))

> **Check.** You should see `Share with at least one failure: 0.05852` and `Formula 1 - 0.98 ** 3: 0.0588`. The simulation is close to the formula, but not equal. With another seed, the share changes a little.

## 8. When independence breaks

All three checks read from the same ticket database. When the database is slow, all three fail together. The next cell simulates this case: for 2 in 100 tickets the database is slow, and then all three checks fail.

> **Predict.** How often do all three checks fail on the same ticket, with independent failures and with one shared cause? Then run the cell.

In [ ]:
random.seed(1)
all_three_independent = 0
all_three_shared = 0
for ticket in range(TICKETS):
    fails = [random.random() < FAIL for check in range(3)]
    if all(fails):
        all_three_independent += 1
    if random.random() < FAIL:       # the database is slow: every check fails
        all_three_shared += 1

print("formula if independent:", round(FAIL ** 3, 6))
print("independent, simulated:", all_three_independent, "of", TICKETS)
print("shared cause, simulated:", all_three_shared, "of", TICKETS)
print("shared cause / formula: ", round(FAIL / FAIL ** 3))

> **Check.** You should see `formula if independent: 8e-06` (8 in a million, Python's short form for 0.000008), `independent, simulated: 1 of 100000` (about 1 in 100,000 is what the formula expects), a shared-cause count close to 2,000 (`1988 of 100000`), and `2500`: with a shared cause, "all three fail" happens 2,500 times more often.

**Worked example: the outage report.** In 5 slow hours, 310 tickets arrived, and all three checks failed on 296 of them. Run the cell. You should see that the independent calculation expects `0.00248` such tickets, and the logs show `296`.

In [ ]:
arrived, all_failed = 310, 296
print("expected if independent:", round(arrived * FAIL ** 3, 5))
print("in the logs:            ", all_failed)
print("share in the logs:      ", round(all_failed / arrived, 3))

> **Check.** You should see a share of `0.955` in the logs. The multiplication was correct arithmetic, but it answered the wrong question.

## 9. Guided practice: count events in the Tuesday table

> **Your turn.** Replace each `...` with a calculation. Use the lists `tickets` and `urgent` from section 1, or the counts from the table: 28 delivery tickets, 2 of them urgent, 8 urgent tickets in all. Then run the check cell.

In [ ]:
p_delivery = 28 / total
p_not_delivery = 1 - p_delivery
p_delivery_urgent = 2 / total
expected_independent = 28 * p_urgent

Run the check cell.

In [ ]:
expect("P(delivery)", p_delivery, 0.28)
expect("P(not delivery)", p_not_delivery, 0.72)
expect("P(delivery and urgent)", p_delivery_urgent, 0.02)
expect("the expected count if independent", expected_independent, 2.24)

The real count is 2, very close to 2.24. So "delivery" and "urgent" look independent in this sample. Compare the payment team: 1.2 expected, 4 real. Write one sentence in your `c03-notes.md`.

## 10. Independent variation: less reliable checks

Change one condition: each check now fails for **5 in 100** tickets. The checks still fail independently.

> **Your turn.** Calculate the chance that all three checks work, the chance that at least one fails, and the number of tickets with a failure in a week of 1,200. Then find the smallest number of checks for which the chance of at least one failure is above 25%. Use the table in section 6. Run the check cell.

In [ ]:
all_work_5 = 0.95 ** 3
at_least_one_5 = 1 - all_work_5
week_5 = 1200 * at_least_one_5
checks_above_25 = min(k for k in range(1, 11) if 1 - 0.95 ** k > 0.25)

Run the check cell.

In [ ]:
expect("all three work", all_work_5, 0.857, tol=0.001)
expect("at least one fails", at_least_one_5, 0.143, tol=0.001)
expect("tickets in a week", week_5, 171, tol=1)
expect("the number of checks", checks_above_25, 6)

> **Your turn: change one thing.** In the simulation cell of section 7, change `FAIL = 0.02` to `FAIL = 0.05` and run it again. The share should be close to your `at_least_one_5`. Then change it back to `0.02`.

A change from 2% to 5% per check is the difference between about 71 and about 171 tickets a week.

## 11. Failure case: multiplying chances that share a cause

A design document says "all three checks fail together for only 8 tickets in a million, so we do not need a fallback". Section 8 showed why this is wrong: the checks share a database.

> **Warning.** Multiply chances only for independent events. Before you multiply, ask: do these events share a cause, such as the same database, the same outage or the same day? If they do, count how often they happen together in the logs.

## Check your understanding and recap

Answer the **knowledge checks** at the end of the [lesson page](https://learning.nextia-ai.com/courses/math/m03/events-and-uncertainty/). They count toward your certificate when you are signed in and enrolled. The notebook does not have them.

In this notebook you counted the Tuesday tickets: P(urgent) = 0.08, its complement 0.92, P(payment and urgent) = 0.04 and P(payment or urgent) = 0.19. Payment and urgent are not independent (1.2 expected, 4 real). With independent checks, at least one of three fails for 5.9% of tickets (14.3% at 5% per check), and a seeded simulation agrees. With a shared cause, "all three fail" is 2,500 times more common than the multiplication says.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Conditional probability](https://learning.nextia-ai.com/courses/math/m03/conditional-probability/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/math/m03/events-and-uncertainty/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The Larkfield tickets are fictional and have no real people.